## Análisis de Datos Meteorológicos con una API Pública
Obtener datos del clima de una ciudad usando la API de Open-Meteo (https://open-meteo.com/) y mostrar el pronóstico para Bilbao.

* Haz una petición a la API para que devuelva el pronóstico de las temperaturas para una semana en Bilbao. Tip: el parámetro correspondiente en el query es `temperature_2m` además de la latitud y la longitud.
* Procesa la respuesta en formato JSON para obtener dos listas: una con la lista de temperaturas y otra con las fechas-horas correspondientes.
* Crea un DataFrame con esas dos columnas: `Hora` y `Temperatura`
* Elige la gráfica adecuada y visualiza el pronóstico de temperaturas.

Para crear la variable `Hora` del DataFrame puedes introducir directamente la lista que se obtiene del campo `data["hourly"]["time"]` o procesar esa lista mediante el siguiente código:

`hours = pd.date_range(start=lista_de_horas, periods=len(temps), freq="H")`

In [ ]:
import requests


# Coordenadas de Kuna en Bilbao
latitude = 43.2540
longitude = -2.9230


url = f"https://api.open-meteo.com/v1/forecast?"  # Aquí vienen el resto de los parámetros

In [ ]:
https://api.open-meteo.com/v1/forecast?latitude=43.254&longitude=-2.923&hourly=temperature_2m&forecast_days=14



In [2]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ---------------------------------------- 0.0/207.7 kB ? eta -:--:--
   ----------------- ---------------------- 92.2/207.7 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 207.7/207.7 kB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/711.0 kB ? eta -:--:--
   --------------------------------------  706.6/711.0 kB 14.8 MB/s eta 0:00:01
   --------------------------------------- 711.0/711.0 kB 11.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/138.8 kB ? eta -:--:--
   ---------------------------------------- 138.8/138.8 kB ? eta 0:00:00
   ---------------------------------------- 0.0/246.3 kB ? eta -:--:--
   ---------------------------------------- 246.3/246.3 kB 7.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------- ------------------------- 0.7/2.0 MB 15.7 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 16.7 MB/s eta 0:00:01
   ------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/70.2 kB ? eta -:--:--
   ----------------------- ---------------- 41.0/70.2 kB 653.6 kB/s eta 0:00:01
   ---------------------------------------- 70.2/70.2 kB 951.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/73.1 kB ? eta -:--:--
   ---------------------------------------- 73.1/73.1 kB ? eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 43.254,
	"longitude": -2.923,
	"hourly": "temperature_2m",
	"forecast_days": 14,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 43.25°N -2.9200000762939453°E
Elevation: 14.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-03-23 00:00:00+00:00       12.340501
1   2026-03-23 01:00:00+00:00       12.140500
2   2026-03-23 02:00:00+00:00       11.840501
3   2026-03-23 03:00:00+00:00       11.490500
4   2026-03-23 04:00:00+00:00       11.140500
..                        ...             ...
331 2026-04-05 19:00:00+00:00        5.536000
332 2026-04-05 20:00:00+00:00        5.386000
333 2026-04-05 21:00:00+00:00        5.286000
334 2026-04-05 22:00:00+00:00        5.186000
335 2026-04-05 23:00:00+00:00        5.136000

[336 rows x 2 columns]


In [ ]:
hourly_dataframe